In [22]:
import yaml
import numpy as np
import matplotlib.pyplot as plt
import numpy as np

def quat_to_rot_matrix(q):
    x, y, z, w = q

    return np.array([
        [1 - 2*(y**2 + z**2),     2*(x*y - z*w),     2*(x*z + y*w)],
        [    2*(x*y + z*w), 1 - 2*(x**2 + z**2),     2*(y*z - x*w)],
        [    2*(x*z - y*w),     2*(y*z + x*w), 1 - 2*(x**2 + y**2)]
    ])

# === 1. Ler YAML ===
with open('telemetria_total.yaml', 'r') as f:
    data = yaml.safe_load(f)

all_points = []
all_matrix = []
# matriz medida parado
R_offset = np.array([
    [-0.9960751 , -0.08848921,  0.00201145],
    [ 0.08828909, -0.99492122, -0.04834051],
    [ 0.00627885, -0.04797319,  0.99882889]
])

# inversa de matriz de rotação = transposta
R_offset_inv = R_offset.T


# === 2. Loop nos dados sincronizados ===
for entry in data[1:]:
    lidar = entry['lidar']
    imu = entry['imu']

    ranges = np.array(lidar['ranges'])
    angle_min = lidar['angle_min']
    angle_increment = lidar['angle_increment']

    # Remove valores inválidos
    valid = np.isfinite(ranges)
    ranges = ranges[valid]

    angles = angle_min + np.arange(len(valid))[valid] * angle_increment

    # === 3. Polar → Cartesiano ===
    x = ranges * np.cos(angles)
    y = ranges * np.sin(angles)
    z = np.zeros_like(x)

    points = np.vstack((x, y, z)).T

    # === 4. Quaternion → Rotação ===
    q = imu['orientation']
    quat = [q['x'], q['y'], q['z'], q['w']]

    R_matrix = quat_to_rot_matrix(quat)
    # em runtime:
    R_matrix = R_offset_inv @ R_matrix
    all_matrix.append(R_matrix)

    # === 5. Transformação homogênea (só rotação aqui) ===
    transformed = (R_matrix @ points.T).T

    all_points.append(transformed)

# Junta tudo
all_points = np.vstack(all_points)

# === 6. Plot ===
plt.figure(figsize=(8,8))
plt.scatter(all_points[:,0], all_points[:,1], s=1)
plt.title("Lidar corrigido pela orientação da IMU")
plt.axis('equal')
plt.grid()
plt.show()

AttributeError: module 'matplotlib' has no attribute 'get_data_path'

In [ ]:
np.round(all_matrix[-1],2)

array([[ 0.95, -0.32, -0.  ],
       [ 0.32,  0.95,  0.  ],
       [ 0.  , -0.  ,  1.  ]])

In [ ]:
np.rad2deg(np.arccos(0.95))

np.float64(18.194872338766785)